In [3]:
import pandas as pd

# Example: Load your data
news_df = pd.read_csv("../data/catalyst_news_2025-06-11.csv")  # columns: 'title', 'description', 'content', 'publishedAt'
stocks_df = pd.read_csv("../data/tickers.csv")    # columns: 'ticker', 'company'

# Combine all textual info for context
news_df["text"] = news_df[["title", "description", "topic"]].fillna("").agg(" ".join, axis=1)

In [4]:
news_df

,publishedAt,source,title,description,url,topic,text
0,2025-06-10T20:00:00Z,GlobeNewswire,Inventiva Announces the Appointment of Renée A...,"Daix (France), New York City (New York, United...",https://www.globenewswire.com/news-release/202...,FDA_Approval,Inventiva Announces the Appointment of Renée A...
1,2025-06-10T15:47:24Z,Levernews.com,FDA Approved Hundreds of Drugs Without Evidenc...,Patients serve as the unwitting guinea pigs — ...,https://www.levernews.com/fda-approved-and-ine...,FDA_Approval,FDA Approved Hundreds of Drugs Without Evidenc...
2,2025-06-10T14:20:00Z,GlobeNewswire,BioPorto A/S Announces the Appointment of a Ne...,"June 10, 2025Announcement no. 16 BioPorto A/S...",https://www.globenewswire.com/news-release/202...,FDA_Approval,BioPorto A/S Announces the Appointment of a Ne...
3,2025-06-10T14:00:00Z,Plos.org,First-line toripalimab plus chemotherapy versu...,Objectives This study aims to evaluate the cos...,https://journals.plos.org/plosone/article?id=1...,FDA_Approval,First-line toripalimab plus chemotherapy versu...
4,2025-06-10T14:00:00Z,Plos.org,Severe cutaneous adverse reactions associated ...,Prostate cancer ranks as the second most preva...,https://journals.plos.org/plosone/article?id=1...,FDA_Approval,Severe cutaneous adverse reactions associated ...
...,...,...,...,...,...,...,...
977,2025-06-10T13:56:00Z,TechSpot,YouTube's relaxed moderation policy allows mor...,The Google-owned site has provided moderators ...,https://www.techspot.com/news/108255-youtube-r...,Short_Squeeze,YouTube's relaxed moderation policy allows mor...
978,2025-06-10T13:55:13Z,Nakedcapitalism.com,“Is the US on the Path to Becoming a Failed St...,"Yes, the US is in the midst of multifaceted in...",https://www.nakedcapitalism.com/2025/06/is-the...,Short_Squeeze,“Is the US on the Path to Becoming a Failed St...
979,2025-06-10T13:49:33Z,Bitcoinist,Snorter Token Next 100x Crypto to Stake as Sta...,Around 34.69M $ETH is currently staked on the ...,https://bitcoinist.com/snorter-token-next-100x...,Short_Squeeze,Snorter Token Next 100x Crypto to Stake as Sta...
980,2025-06-10T13:30:39Z,newsBTC,Bitcoin’s $110K Sprint Coincides With Record-L...,Bitcoin’s price shook off last week’s dip and ...,http://www.newsbtc.com/news/bitcoin/bitcoins-1...,Short_Squeeze,Bitcoin’s $110K Sprint Coincides With Record-L...


In [8]:
stocks_df

,ticker,name,exchange
0,AACB,Artius II Acquisition Inc. Class A Ordinary Sh...,NASDAQ
1,AACBR,Artius II Acquisition Inc. Rights,NASDAQ
2,AACBU,Artius II Acquisition Inc. Units,NASDAQ
3,AACG,ATA Creativity Global American Depositary Shares,NASDAQ
4,AACIU,Armada Acquisition Corp. II Units,NASDAQ
...,...,...,...
6934,XPL,Solitario Resources Corp. Common Stock,AMEX
6935,XTNT,Xtant Medical Holdings Inc. Common Stock,AMEX
6936,YCBD,cbdMD Inc. Common Stock,AMEX
6937,ZDGE,Zedge Inc. Class B Common Stock,AMEX


In [6]:
import spacy
nlp = spacy.load("en_core_web_trf")  # Load the transformer-based model
def extract_entities(text):
    """Extract named entities from text using spaCy."""
    doc = nlp(text)
    return [(ent.text, ent.label_) for ent in doc.ents]

e:\Github Projects 2025\1.Github\quant-trading-system\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
from rapidfuzz import process, fuzz

# Create a lowercase set for faster match
company_names = stocks_df['name'].tolist()

def match_entities(text):
    doc = nlp(text)
    found_matches = set()

    for ent in doc.ents:
        if ent.label_ in ["ORG", "PRODUCT", "PERSON", "GPE"]:  # Common labels for company mentions
            match, score, _ = process.extractOne(ent.text, company_names, scorer=fuzz.token_sort_ratio)
            if score >= 85:  # Adjust threshold as needed
                print(f"Matched: {ent.text} -> {match} (Score: {score})")
                # Add the match to the set to avoid duplicates
                found_matches.add(match)
    
    return list(found_matches)

# Apply to your news DataFrame
news_df["matched_companies"] = news_df["text"].apply(match_entities)

e:\Github Projects 2025\1.Github\quant-trading-system\.venv\lib\site-packages\thinc\shims\pytorch.py:109: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(self._mixed_precision):


Matched: Rocket Pharmaceuticals, Inc. -> Rocket Pharmaceuticals Inc. Warrant (Score: 85.71428571428572)
Matched: Rocket Pharmaceuticals, Inc. -> Rocket Pharmaceuticals Inc. Warrant (Score: 85.71428571428572)
Matched: Abacus Global Management, Inc. -> Abacus Global Management Inc. Warrant (Score: 86.56716417910448)
Matched: Altria Group, Inc. -> Altria Group Inc. (Score: 97.14285714285714)
Matched: Altria Group, Inc. -> Altria Group Inc. (Score: 97.14285714285714)
Matched: Yum! Brands, Inc. -> Yum! Brands Inc. (Score: 96.96969696969697)
Matched: Yum! Brands, Inc. -> Yum! Brands Inc. (Score: 96.96969696969697)
Matched: Public Service Enterprise Group Incorporated -> Public Service Enterprise Group Incorporated Common Stock (Score: 87.12871287128714)
Matched: Public Service Enterprise Group Incorporated -> Public Service Enterprise Group Incorporated Common Stock (Score: 87.12871287128714)
Matched: AMETEK, Inc. -> AMETEK Inc. (Score: 95.65217391304348)
Matched: AMETEK, Inc. -> AMETEK Inc.

In [10]:
news_df

,publishedAt,source,title,description,url,topic,text,matched_companies
0,2025-06-10T20:00:00Z,GlobeNewswire,Inventiva Announces the Appointment of Renée A...,"Daix (France), New York City (New York, United...",https://www.globenewswire.com/news-release/202...,FDA_Approval,Inventiva Announces the Appointment of Renée A...,[]
1,2025-06-10T15:47:24Z,Levernews.com,FDA Approved Hundreds of Drugs Without Evidenc...,Patients serve as the unwitting guinea pigs — ...,https://www.levernews.com/fda-approved-and-ine...,FDA_Approval,FDA Approved Hundreds of Drugs Without Evidenc...,[]
2,2025-06-10T14:20:00Z,GlobeNewswire,BioPorto A/S Announces the Appointment of a Ne...,"June 10, 2025Announcement no. 16 BioPorto A/S...",https://www.globenewswire.com/news-release/202...,FDA_Approval,BioPorto A/S Announces the Appointment of a Ne...,[]
3,2025-06-10T14:00:00Z,Plos.org,First-line toripalimab plus chemotherapy versu...,Objectives This study aims to evaluate the cos...,https://journals.plos.org/plosone/article?id=1...,FDA_Approval,First-line toripalimab plus chemotherapy versu...,[]
4,2025-06-10T14:00:00Z,Plos.org,Severe cutaneous adverse reactions associated ...,Prostate cancer ranks as the second most preva...,https://journals.plos.org/plosone/article?id=1...,FDA_Approval,Severe cutaneous adverse reactions associated ...,[]
...,...,...,...,...,...,...,...,...
977,2025-06-10T13:56:00Z,TechSpot,YouTube's relaxed moderation policy allows mor...,The Google-owned site has provided moderators ...,https://www.techspot.com/news/108255-youtube-r...,Short_Squeeze,YouTube's relaxed moderation policy allows mor...,[]
978,2025-06-10T13:55:13Z,Nakedcapitalism.com,“Is the US on the Path to Becoming a Failed St...,"Yes, the US is in the midst of multifaceted in...",https://www.nakedcapitalism.com/2025/06/is-the...,Short_Squeeze,“Is the US on the Path to Becoming a Failed St...,[]
979,2025-06-10T13:49:33Z,Bitcoinist,Snorter Token Next 100x Crypto to Stake as Sta...,Around 34.69M $ETH is currently staked on the ...,https://bitcoinist.com/snorter-token-next-100x...,Short_Squeeze,Snorter Token Next 100x Crypto to Stake as Sta...,[]
980,2025-06-10T13:30:39Z,newsBTC,Bitcoin’s $110K Sprint Coincides With Record-L...,Bitcoin’s price shook off last week’s dip and ...,http://www.newsbtc.com/news/bitcoin/bitcoins-1...,Short_Squeeze,Bitcoin’s $110K Sprint Coincides With Record-L...,[]
